# Experimental Run — GEO Audit

Pipeline experimental que mide la visibilidad GEO de programamos.es.

**Prerequisito**: Haber ejecutado `00_discover_competitors.ipynb` para tener
el vectorstore congelado en `data/frozen_vectorstore/`.

**Que hace cada run**:
1. Carga el vectorstore congelado de competidores
2. Scrapea el contenido ACTUAL de programamos.es
3. Genera embeddings solo del target y los mezcla con los congelados
4. Para cada query: retrieve → RAG Judge (JSON) → Citation Extractor
5. Guarda scorecard JSON en `data/geo/experimental/`

Ver ADR-006 en `docs/DECISIONS.md`

In [ ]:
# === Setup del entorno ===
import os
import sys
import json
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO, format='%(name)s — %(message)s')

if os.path.exists('/kaggle'):
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    os.environ['OPENAI_API_KEY'] = secrets.get_secret('OPENAI_API_KEY')
    os.environ['USER_AGENT'] = 'GeoAuditBot/1.0 (TFG Research)'
    sys.path.insert(0, '/kaggle/input/tfg')
else:
    from dotenv import load_dotenv
    load_dotenv()
    sys.path.insert(0, os.path.abspath('..'))

# Resolver PROJECT_ROOT para rutas de archivos (no depender del CWD)
from src.config import PROJECT_ROOT
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# === Cargar configuracion ===
from src.config import load_experiment_config, get_all_queries

config = load_experiment_config()
queries = get_all_queries()
target_url = config['target_url']
target_brand = config['target_brand']

RUN_ID = datetime.now().strftime('run_%Y%m%d_%H%M%S')
print(f"Run: {RUN_ID}")
print(f"Target: {target_url}")
print(f"Queries: {len(queries)}")

In [ ]:
# === Cargar vectorstore congelado de competidores ===
from src.processing.embeddings import create_embeddings
from langchain_community.vectorstores import FAISS

embeddings = create_embeddings(config)
vs_path = PROJECT_ROOT / 'data' / 'frozen_vectorstore'

if not vs_path.exists():
    raise FileNotFoundError(
        f"Vectorstore congelado no encontrado en {vs_path}/\n"
        "Ejecuta 00_discover_competitors.ipynb primero."
    )

competitor_vs = FAISS.load_local(
    str(vs_path), embeddings, allow_dangerous_deserialization=True
)
print(f"Vectorstore congelado cargado: {competitor_vs.index.ntotal} vectores")

In [ ]:
# === Scrape contenido actual de programamos.es ===
from src.processing.html_processor import StructuredWebLoader
from src.processing.chunker import TokenAwareChunker

loader = StructuredWebLoader(target_url)
target_docs = loader.load()

if not target_docs:
    raise RuntimeError(f"No se pudo descargar {target_url}")

for d in target_docs:
    d.metadata['is_target'] = True

chunker = TokenAwareChunker(config)
target_chunks = chunker.chunk_documents(target_docs)
print(f"Target: {len(target_chunks)} chunks de programamos.es")

In [ ]:
# === Merge vectores: target actual + competidores congelados ===
target_vs = FAISS.from_documents(target_chunks, embeddings)
target_vs.merge_from(competitor_vs)

retriever = target_vs.as_retriever(
    search_kwargs={'k': config['retrieval']['top_k']}
)
print(f"Vectorstore combinado: {target_vs.index.ntotal} vectores total")

In [ ]:
# === Ejecutar RAG Judge + Citation Extractor para cada query ===
from src.rag.judge import RAGJudge
from src.rag.citation_extractor import CitationExtractor

judge = RAGJudge(config)
extractor = CitationExtractor(target_url, target_brand)

results = []
for i, query in enumerate(queries, 1):
    print(f"\n[{i}/{len(queries)}] {query}")
    
    # Retrieve
    retrieved = retriever.invoke(query)
    
    # Judge
    try:
        judge_output = judge.generate_answer(query, retrieved)
        metrics = extractor.extract_metrics(judge_output)
        
        result = {
            'query': query,
            'judge_output': judge_output,
            'metrics': metrics,
            'retrieved_urls': [d.metadata.get('source_url', '') for d in retrieved],
        }
        results.append(result)
        
        visible = '✓' if metrics['is_visible'] else '✗'
        print(f"  {visible} Visible={metrics['is_visible']}, SoM={metrics['som']}%, Rank={metrics['first_citation_rank']}")
    except Exception as e:
        print(f"  ERROR: {e}")
        results.append({'query': query, 'error': str(e)})

In [ ]:
# === Guardar scorecard ===
run_dir = PROJECT_ROOT / 'data' / 'geo' / 'experimental' / RUN_ID
run_dir.mkdir(parents=True, exist_ok=True)

# Scorecard resumen
valid_results = [r for r in results if 'metrics' in r]
visible_count = sum(1 for r in valid_results if r['metrics']['is_visible'])
ranked_results = [r for r in valid_results if r['metrics']['first_citation_rank'] is not None]

scorecard = {
    'run_id': RUN_ID,
    'timestamp': datetime.now().isoformat(),
    'target_url': target_url,
    'target_brand': target_brand,
    'n_queries': len(queries),
    'n_successful': len(valid_results),
    'aggregate_metrics': {
        'visibility_rate': (visible_count / len(valid_results) * 100) if valid_results else 0,
        'avg_som': sum(r['metrics']['som'] for r in valid_results) / len(valid_results) if valid_results else 0,
        'avg_rank': sum(r['metrics']['first_citation_rank'] for r in ranked_results) / len(ranked_results) if ranked_results else None,
    },
    'token_usage': judge.get_token_usage_summary(),
    'config_snapshot': config,
}

# Guardar archivos
with open(run_dir / 'scorecard.json', 'w', encoding='utf-8') as f:
    json.dump(scorecard, f, ensure_ascii=False, indent=2)

with open(run_dir / 'raw_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Resultados guardados en {run_dir}/")
print(f"\n=== RESUMEN ===")
print(f"Visibilidad: {scorecard['aggregate_metrics']['visibility_rate']:.1f}%")
print(f"SoM promedio: {scorecard['aggregate_metrics']['avg_som']:.1f}%")
avg_rank = scorecard['aggregate_metrics']['avg_rank']
print(f"Rank promedio: {f'{avg_rank:.1f}' if avg_rank else 'N/A (no citado)'}")
print(f"Tokens usados: {scorecard['token_usage']}")

In [ ]:
# === Detalle por query ===
print(f"{'Query':<70} {'Vis':>4} {'SoM':>6} {'Rank':>5}")
print('-' * 90)
for r in results:
    if 'metrics' in r:
        m = r['metrics']
        vis = 'SI' if m['is_visible'] else 'NO'
        rank = str(m['first_citation_rank']) if m['first_citation_rank'] else '-'
        print(f"{r['query'][:68]:<70} {vis:>4} {m['som']:>5.1f}% {rank:>5}")
    else:
        print(f"{r['query'][:68]:<70} {'ERR':>4}")